In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,Embedding,Dense,LSTM,Dropout,Bidirectional
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer


import warnings
warnings.filterwarnings("ignore")

In [76]:
df = pd.read_csv("sms_spam_collection.csv")
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [77]:
df.shape

(5572, 2)

In [78]:
df.columns

Index(['label', 'message'], dtype='object')

In [79]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [80]:
df.isnull().sum()

label      0
message    0
dtype: int64

In [81]:
df.duplicated().sum()

np.int64(403)

In [82]:
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [83]:
df.shape

(5169, 2)

In [84]:
df["message"][0]

'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...'

In [85]:
df["message"][2]

"Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"

In [86]:
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

In [87]:
df["message"] = df["message"].apply(remove_punc)
df.head()

,label,message
0,ham,Go until jurong point crazy Available only in ...
1,ham,Ok lar Joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor U c already then say
4,ham,Nah I dont think he goes to usf he lives aroun...


In [88]:
def remove_num(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new+=i
    return new

In [89]:
df["message"] = df["message"].apply(remove_num)
df.head()

,label,message
0,ham,Go until jurong point crazy Available only in ...
1,ham,Ok lar Joking wif u oni
2,spam,Free entry in a wkly comp to win FA Cup final...
3,ham,U dun say so early hor U c already then say
4,ham,Nah I dont think he goes to usf he lives aroun...


In [90]:
def remove_emoj(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new+=i
    return new

In [91]:
df["message"] = df["message"].apply(remove_emoj)
df.head()

,label,message
0,ham,Go until jurong point crazy Available only in ...
1,ham,Ok lar Joking wif u oni
2,spam,Free entry in a wkly comp to win FA Cup final...
3,ham,U dun say so early hor U c already then say
4,ham,Nah I dont think he goes to usf he lives aroun...


In [92]:
def tolower(txt):
    return txt.lower()

In [93]:
df["message"] = df["message"].apply(tolower)
df.head()

,label,message
0,ham,go until jurong point crazy available only in ...
1,ham,ok lar joking wif u oni
2,spam,free entry in a wkly comp to win fa cup final...
3,ham,u dun say so early hor u c already then say
4,ham,nah i dont think he goes to usf he lives aroun...


In [94]:
X = df[["message"]]
y = df[["label"]]

In [95]:
label_en = LabelEncoder()

y["label"] = label_en.fit_transform(df["label"])
y.head()

,label
0,0
1,0
2,1
3,0
4,0


In [96]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [97]:
Max_len = 100
Max_words = 10000
embedding_dim = 100

In [98]:
tokenizer = Tokenizer(num_words=Max_words,oov_token="<OOV>")
tokenizer.fit_on_texts(X_train["message"])
train_seq = tokenizer.texts_to_sequences(X_train["message"])
train_pad = pad_sequences(train_seq,maxlen=Max_len,padding="post",truncating="post")
test_seq = tokenizer.texts_to_sequences(X_test["message"])
test_pad = pad_sequences(test_seq,maxlen=Max_len,padding="post",truncating="post")

In [99]:
model = Sequential([
    Embedding(input_dim=Max_words,output_dim=embedding_dim,input_length=Max_len),
    Bidirectional(LSTM(64,return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(32,return_sequences=False)),
    Dropout(0.2),
    Dense(64,activation="relu"),
    Dense(32,activation="relu"),
    Dense(1,activation="sigmoid")
])


In [100]:
model.compile(loss="binary_crossentropy",optimizer="adam",metrics=["accuracy"])
# model.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True)
his = model.fit(train_pad,y_train,validation_data=(test_pad,y_test),epochs=10,batch_size=64)


Epoch 1/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 23s 213ms/step - accuracy: 0.9139 - loss: 0.2590 - val_accuracy: 0.9710 - val_loss: 0.0970
Epoch 2/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 12s 190ms/step - accuracy: 0.9852 - loss: 0.0629 - val_accuracy: 0.9836 - val_loss: 0.0658
Epoch 3/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 15s 230ms/step - accuracy: 0.9925 - loss: 0.0331 - val_accuracy: 0.9807 - val_loss: 0.0728
Epoch 4/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 17s 261ms/step - accuracy: 0.9944 - loss: 0.0211 - val_accuracy: 0.9826 - val_loss: 0.0639
Epoch 5/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 15s 230ms/step - accuracy: 0.9976 - loss: 0.0086 - val_accuracy: 0.9865 - val_loss: 0.0654
Epoch 6/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 18s 269ms/step - accuracy: 0.9990 - loss: 0.0055 - val_accuracy: 0.9845 - val_loss: 0.0789
Epoch 7/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 18s 232ms/step - accuracy: 0.9998 - loss: 0.0012 - val_accuracy: 0.9855 - val_loss: 0.0764
Epoch 8/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 16s 252ms/step - accuracy: 0.9995 - loss: 0.0015 - val_accu

In [101]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 100)       │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 100, 128)       │        84,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 100, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,395,909 (12.95 MB)

 Trainable params: 1,131,969 (4.32 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,263,940 (8.64 MB)